In [ ]:
import pandas as pd
import struct
import datetime
from plotly.subplots import make_subplots
import numpy as np
import plotly.graph_objects as go
from dotenv import load_dotenv
import os
from scipy.signal import medfilt
import math

In [ ]:
QUERY = """
select 
    lower(hex(bytes)) as packets_hex,
    bytes as packet
from packets 
where (lower(hex(bytes)) like "aa6400a1%" or lower(hex(bytes)) like "aa5c00f0%") and lower(hex(uuid)) = "610800058d6d82b8614a1c8cb0f8dcc6"
"""

In [ ]:
DATABASE_URL = os.getenv("DATABASE_URL").replace("sqlite://", "sqlite:///../")
df = pd.read_sql(QUERY, DATABASE_URL)

In [ ]:
MAX_DAYS = 120  # Limit to last N days to prevent crashes on large datasets

df["datetime"] = pd.to_datetime(df["packet"].apply(lambda data: struct.unpack('<I', data[11:15])[0]), unit="s")
df = df.sort_values("datetime", ascending=True)

# Filter to last MAX_DAYS
cutoff = df["datetime"].max() - pd.Timedelta(days=MAX_DAYS)
df = df[df["datetime"] >= cutoff].reset_index(drop=True)
print(f"Loaded {len(df)} readings ({MAX_DAYS}-day window: {df['datetime'].min().date()} to {df['datetime'].max().date()})")

df['date'] = df['datetime'].dt.date
df['time'] = df['datetime'].dt.time
df["bpm"] = df["packet"].apply(lambda data: data[21])

In [ ]:
def parse_rr(packet: bytes) -> list:
    rr_count = packet[22]
    packet = packet[23:]
    
    rr = []
    for _ in range(4):
        rr_value = struct.unpack('<H', packet[:2])[0]
        packet = packet[2:]
        if rr_value != 0:
            rr.append(rr_value)

    if len(rr) != rr_count:
        raise ValueError("Invalid data")
    return rr

df["rr"] = df["packet"].apply(parse_rr)

In [ ]:
df["stage"] = df["packet"].apply(lambda data: struct.unpack('<I', data[31:35])[0])

In [ ]:

# The combined 32-bit "stage" value at bytes [31:35] is actually two PPG sensor fields:
#   bytes [31:33] = ppg_flags  (u16 LE) — lower 16 bits of the combined u32
#   bytes [33:35] = ppg_green  (u16 LE) — upper 16 bits of the combined u32
# So:  stage == (ppg_green << 16) | ppg_flags
df["ppg_flags"] = df["packet"].apply(lambda data: struct.unpack('<H', data[31:33])[0])
df["ppg_green"] = df["packet"].apply(lambda data: struct.unpack('<H', data[33:35])[0])

reconstructed = (df["ppg_green"].to_numpy().astype("int64") << 16) | df["ppg_flags"].to_numpy().astype("int64")
assert (reconstructed == df["stage"].to_numpy()).all(), "Decomposition sanity check failed"
print("stage == (ppg_green << 16) | ppg_flags  ✓")


In [ ]:
def plot_heart_rate(days, column, plot_count = 2):
    for index, day_data in enumerate(days):
        # Create a figure with secondary y-axes
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        # Add heart rate trace to primary y-axis
        fig.add_trace(
            go.Scatter(x=day_data['datetime'], y=day_data['bpm'], mode='lines', name='Heart Rate (BPM)', line=dict(color='blue')),
            secondary_y=False,
        )

        # Add x trace to secondary y-axis
        fig.add_trace(
            go.Scatter(x=day_data['datetime'], y=day_data[column], mode='lines', name='X', line=dict(color='orange')),
            secondary_y=True,
        )

        # Update layout for titles and axes
        fig.update_layout(
            title=f"Heart Rate and XYZ from {day_data['datetime'].iloc[0].date()} Noon to Next Day Noon",
            xaxis_title="Time",
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        )

        # Update x-axis for time formatting
        fig.update_xaxes(tickformat='%H:%M')

        # Update y-axes titles
        fig.update_yaxes(title_text="Heart Rate (BPM)", secondary_y=False)
        fig.update_yaxes(title_text="XYZ Values", secondary_y=True)

        # Show the plot
        fig.show()

        if index == plot_count:
            break

In [ ]:
def filter_noon_to_noon(df):
    # Group the data by date
    days = []
    unique_dates = df['date'].unique()
    
    for date in unique_dates:
        # Define noon of the current day and noon of the next day
        start_noon = pd.Timestamp(datetime.datetime.combine(date, datetime.time(12, 0)))
        end_noon = start_noon + pd.Timedelta(days=1)
        
        # Filter data between the start and end noon
        day_data = df[(df['datetime'] >= start_noon) & (df['datetime'] < end_noon)].copy()
        if not day_data.empty:
            days.append(day_data)

    return days

In [ ]:
def remove_spikes_row(row, window_size=3, threshold=3):
    # Apply median filter
    filtered = medfilt(row, kernel_size=window_size)
    # Identify spikes
    deviation = np.abs(row - filtered)
    is_spike = deviation > threshold * np.std(row)
    # Replace spikes with filtered values
    smoothed = row.copy()
    smoothed[is_spike] = filtered[is_spike]
    return smoothed

In [ ]:
df["stage"] = remove_spikes_row(df["stage"])

In [ ]:
df.loc[df["stage"] < 500_000_000, "stage_category"] = 0
df.loc[(df["stage"] >= 500_000_000) & (df["stage"] < 1000_000_000), "stage_category"] = 1
df.loc[(df["stage"] >= 1000_000_000) & (df["stage"] < 1500_000_000), "stage_category"] = 2
df.loc[df["stage"] > 1500_000_000, "stage_category"] = 3
df["stage_category"] = df["stage_category"].astype("int64")

In [ ]:
days = filter_noon_to_noon(df)

In [ ]:
plot_heart_rate(days, "stage")

In [ ]:
plot_heart_rate(days, "stage_category")

In [ ]:
ACTIVITY_DURATION = 600 * 2


def precompute_hrv(df):
    """Pre-compute vectorized RMSSD series once for fast HRV lookups.

    RMSSD = sqrt(mean(successive_diff^2)) over rolling windows.
    Instead of calling a Python function per window (extremely slow),
    we compute all successive diffs, square them, and use pandas
    vectorized rolling mean.
    """
    rr_exploded = df[["datetime", "rr"]].explode("rr").dropna(subset=["rr"])
    if rr_exploded.empty:
        return np.array([]), np.array([])
    rr_vals = rr_exploded["rr"].astype(float).values
    rr_dts = rr_exploded["datetime"].values

    diffs_sq = np.diff(rr_vals) ** 2
    rmssd = np.sqrt(pd.Series(diffs_sq).rolling(299, min_periods=10).mean().values)
    return rr_dts[1:], rmssd  # shifted by 1 due to diff


def annotate_periods(periods, df, hrv_dts, hrv_vals):
    """Add BPM and HRV stats using searchsorted instead of per-row apply."""
    if periods.empty:
        return periods

    dt = df["datetime"].values
    bpm = df["bpm"].values
    starts = periods["start"].values
    ends = periods["end"].values

    # BPM stats via searchsorted — O(n log m) instead of O(n * m)
    s_idx = np.searchsorted(dt, starts, side="left")
    e_idx = np.searchsorted(dt, ends, side="right")

    avg_b, min_b, max_b = [], [], []
    for s, e in zip(s_idx, e_idx):
        seg = bpm[s:e]
        if len(seg):
            avg_b.append(int(round(np.mean(seg))))
            min_b.append(int(np.min(seg)))
            max_b.append(int(np.max(seg)))
        else:
            avg_b.append(0)
            min_b.append(0)
            max_b.append(0)

    periods["avg_bpm"] = avg_b
    periods["min_bpm"] = min_b
    periods["max_bpm"] = max_b

    # HRV stats via searchsorted on pre-computed RMSSD
    if len(hrv_dts):
        hs = np.searchsorted(hrv_dts, starts, side="left")
        he = np.searchsorted(hrv_dts, ends, side="right")
        avg_h, min_h, max_h = [], [], []
        for s, e in zip(hs, he):
            seg = hrv_vals[s:e]
            valid = seg[~np.isnan(seg)]
            if len(valid):
                avg_h.append(int(np.mean(valid)))
                min_h.append(int(np.min(valid)))
                max_h.append(int(np.max(valid)))
            else:
                avg_h.append(0)
                min_h.append(0)
                max_h.append(0)
        periods["avg_hrv"] = avg_h
        periods["min_hrv"] = min_h
        periods["max_hrv"] = max_h
    else:
        periods[["avg_hrv", "min_hrv", "max_hrv"]] = 0

    return periods


def detect_stages(df, stage):
    """Detect contiguous stage periods with vectorized merge."""
    cat = df["stage_category"].values
    dt = df["datetime"].values
    is_stage = cat == stage

    # Find contiguous runs using diff on padded boolean array
    padded = np.concatenate([[False], is_stage, [False]])
    changes = np.diff(padded.astype(int))
    run_starts = np.where(changes == 1)[0]
    run_ends = np.where(changes == -1)[0] - 1

    if len(run_starts) == 0:
        return pd.DataFrame(
            columns=["start", "end", "duration",
                      "avg_bpm", "min_bpm", "max_bpm",
                      "avg_hrv", "min_hrv", "max_hrv"]
        )

    start_times = dt[run_starts]
    end_times = dt[run_ends]

    # Single-pass merge of gaps < ACTIVITY_DURATION
    gap_thresh = np.timedelta64(ACTIVITY_DURATION, "s")
    ms, me = [start_times[0]], [end_times[0]]
    for i in range(1, len(start_times)):
        if start_times[i] - me[-1] < gap_thresh:
            me[-1] = end_times[i]
        else:
            ms.append(start_times[i])
            me.append(end_times[i])

    ms, me = np.array(ms), np.array(me)

    # Filter minimum duration
    dur_s = (me - ms) / np.timedelta64(1, "s")
    keep = dur_s >= ACTIVITY_DURATION

    stages = pd.DataFrame({
        "start": pd.to_datetime(ms[keep]),
        "end": pd.to_datetime(me[keep]),
    })
    if stages.empty:
        return stages

    stages["duration"] = (
        (stages["end"] - stages["start"]).dt.total_seconds() / 3600
    ).round(2)
    return annotate_periods(stages, df, _hrv_dts, _hrv_vals)


# Pre-compute HRV once (reused by detect_stages and detect_sleep_from_gravity)
_hrv_dts, _hrv_vals = precompute_hrv(df)
print(f"Pre-computed {len(_hrv_vals):,} RMSSD values")

In [ ]:
sleep_df = detect_stages(df, 2)
sleep_df

In [ ]:

# ppg_green over time — should look identical to the original "stage" plot
# but it's just the upper 16 bits of the combined value
plot_heart_rate(days, "ppg_green")


In [ ]:

thresholds = {
    "Inactive/Active": 500_000_000 >> 16,   # 7629
    "Active/Sleep":    1_000_000_000 >> 16,  # 15258
    "Sleep/Awake":     1_500_000_000 >> 16,  # 22888
}

# Compare ppg_green vs ppg_flags distributions by stage category
fig = make_subplots(rows=1, cols=2, subplot_titles=("ppg_green by stage", "ppg_flags by stage"))

colors = {0: "gray", 1: "orange", 2: "blue", 3: "green"}
labels = {0: "Inactive", 1: "Active", 2: "Sleep", 3: "Awake"}

for cat in sorted(df["stage_category"].unique()):
    subset = df[df["stage_category"] == cat]
    fig.add_trace(go.Box(y=subset["ppg_green"], name=labels[cat],
                         marker_color=colors[cat], showlegend=True), row=1, col=1)
    fig.add_trace(go.Box(y=subset["ppg_flags"], name=labels[cat],
                         marker_color=colors[cat], showlegend=False), row=1, col=2)

# Add threshold lines on ppg_green plot
for label, t in thresholds.items():
    fig.add_hline(y=t, line_dash="dash", line_color="red", row=1, col=1,
                  annotation_text=label, annotation_position="right")

fig.update_layout(title="ppg_green vs ppg_flags: which field separates activity stages?",
                  height=500)
fig.update_yaxes(title_text="ppg_green value", row=1, col=1)
fig.update_yaxes(title_text="ppg_flags value", row=1, col=2)
fig.show()


In [ ]:

# Quantify how much ppg_flags actually matters
ppg_flags_max = df["ppg_flags"].max()
range_width = 500_000_000
print(f"ppg_flags observed max:  {ppg_flags_max}  ({ppg_flags_max / range_width:.4%} of one activity range width)")
print(f"ppg_flags theoretical max: 65535  ({65535 / range_width:.4%} of one activity range width)")
print()
print("ppg_green thresholds:")
for label, t in thresholds.items():
    print(f"  {label}: {t}")
print()

# Derive stage_category from ppg_green alone and compare
df["stage_category_ppg_green"] = 0
df.loc[df["ppg_green"] >= thresholds["Inactive/Active"], "stage_category_ppg_green"] = 1
df.loc[df["ppg_green"] >= thresholds["Active/Sleep"],    "stage_category_ppg_green"] = 2
df.loc[df["ppg_green"] >= thresholds["Sleep/Awake"],     "stage_category_ppg_green"] = 3

match_rate = (df["stage_category"] == df["stage_category_ppg_green"]).mean()
print(f"ppg_green alone matches full stage category: {match_rate:.2%} of readings")
mismatch = df[df["stage_category"] != df["stage_category_ppg_green"]][["ppg_flags", "ppg_green", "stage", "stage_category", "stage_category_ppg_green"]]
print(f"Mismatches: {len(mismatch)} rows")
if len(mismatch):
    display(mismatch.head(10))


## Which field drives activity classification?

The `stage` u32 = `(ppg_green << 16) | ppg_flags`. Because `ppg_green` occupies the **upper 16 bits**, it determines which activity range the value falls in. `ppg_flags` (lower 16 bits) can shift the combined value by at most 65535 out of a 500 000 000-wide range — only **0.013%** — so it never changes the category on its own.

| Activity | Combined range | ppg_green range |
|---|---|---|
| Inactive | 0 – 500 M | 0 – 7629 |
| Active | 500 M – 1 B | 7629 – 15 258 |
| **Sleep** | **1 B – 1.5 B** | **15 258 – 22 888** |
| Awake | ≥ 1.5 B | ≥ 22 888 |


In [ ]:
exercises = detect_stages(df, 1)
exercises

## Gravity-based sleep detection

Each V12 packet contains an accelerometer gravity vector `[gx, gy, gz]` at bytes `[40:52]` (3 × f32 LE). The magnitude should be ~1 g since it measures static orientation.

The key signal for sleep detection is the **rate of change** between consecutive readings: if the wrist barely moved in the last minute, the delta will be near zero (sleeping); if the person was active, the delta will be large.


In [ ]:

# Parse gravity vector from raw packet bytes [40:52] (data offset 33:45 + 7 byte header)
def parse_gravity(packet: bytes):
    try:
        gx = struct.unpack('<f', packet[40:44])[0]
        gy = struct.unpack('<f', packet[44:48])[0]
        gz = struct.unpack('<f', packet[48:52])[0]
        return gx, gy, gz
    except struct.error:
        return float('nan'), float('nan'), float('nan')

gravity = df["packet"].apply(parse_gravity)
df["gx"] = gravity.apply(lambda t: t[0])
df["gy"] = gravity.apply(lambda t: t[1])
df["gz"] = gravity.apply(lambda t: t[2])

df["gravity_magnitude"] = np.sqrt(df["gx"]**2 + df["gy"]**2 + df["gz"]**2)
print(f"Gravity magnitude — mean: {df['gravity_magnitude'].mean():.3f} g  "
      f"std: {df['gravity_magnitude'].std():.3f} g  "
      f"(expect ~1.0 g)")

# Rate of change between consecutive readings (~1 min apart) = wrist movement proxy
df["gravity_delta"] = np.sqrt(
    df["gx"].diff()**2 +
    df["gy"].diff()**2 +
    df["gz"].diff()**2
)
df["gravity_delta_smooth"] = df["gravity_delta"].rolling(5, center=True, min_periods=1).median()

print(f"\ngravity_delta — mean: {df['gravity_delta'].mean():.4f}  "
      f"median: {df['gravity_delta'].median():.4f}  "
      f"max: {df['gravity_delta'].max():.4f}")
for cat, label in [(0, "Inactive"), (1, "Active"), (2, "Sleep"), (3, "Awake")]:
    subset = df[df["stage_category"] == cat]["gravity_delta"]
    if len(subset):
        print(f"  {label:8s}: median={subset.median():.4f}  mean={subset.mean():.4f}  p90={subset.quantile(0.9):.4f}")


In [ ]:

plot_heart_rate(filter_noon_to_noon(df), "gravity_delta_smooth")


In [ ]:

# Distribution of gravity_delta per stage — does it separate cleanly?
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("gravity_delta distribution by stage",
                                    "gravity_delta vs ppg_green scatter"))

colors = {0: "gray", 1: "orange", 2: "blue", 3: "green"}
labels = {0: "Inactive", 1: "Active", 2: "Sleep", 3: "Awake"}

for cat in sorted(df["stage_category"].unique()):
    subset = df[df["stage_category"] == cat]
    fig.add_trace(go.Box(y=subset["gravity_delta"], name=labels[cat],
                         marker_color=colors[cat]), row=1, col=1)
    fig.add_trace(go.Scatter(x=subset["gravity_delta"], y=subset["ppg_green"],
                             mode="markers", name=labels[cat],
                             marker=dict(color=colors[cat], size=4, opacity=0.5),
                             showlegend=False), row=1, col=2)

fig.update_layout(title="gravity_delta: can it replace ppg_green for stage detection?", height=500)
fig.update_yaxes(title_text="gravity_delta (g)", row=1, col=1)
fig.update_xaxes(title_text="gravity_delta (g)", row=1, col=2)
fig.update_yaxes(title_text="ppg_green", row=1, col=2)
fig.show()


## Gravity sleep detection algorithm

Sleep p90 is 0.0049 g — sleep periods are *extremely* still compared to any other stage.
The algorithm:
1. **Still** — reading delta below threshold (0.01 g)
2. **Still window** — rolling 15-min fraction of still readings ≥ 0.7
3. **Merge** — bridge gaps < 20 min (brief awakenings / rolling over)
4. **Filter** — keep only periods ≥ 60 min


In [ ]:
def detect_sleep_from_gravity(
    df,
    delta_threshold=0.01,
    still_fraction=0.70,
    window_minutes=15,
    max_gap_minutes=20,
    min_sleep_minutes=60,
):
    df = df.copy().reset_index(drop=True)

    # --- 1. per-reading stillness ---
    is_still = (df["gravity_delta_smooth"].values < delta_threshold).astype(float)

    # --- 2. rolling still-fraction ---
    interval_min = np.median(
        np.diff(df["datetime"].values).astype("timedelta64[s]").astype(float)
    ) / 60
    window = max(3, round(window_minutes / interval_min))
    still_frac = (
        pd.Series(is_still).rolling(window, center=True, min_periods=1).mean().values
    )

    # --- 3. sleep-candidate mask ---
    sleep_win = still_frac >= still_fraction

    # --- 4. contiguous runs, breaking on data gaps ---
    dt = df["datetime"].values
    gap_break = np.zeros(len(dt), dtype=bool)
    gap_break[1:] = np.diff(dt) > np.timedelta64(max_gap_minutes, "m")

    # Build run boundaries: new run on sleep_win change or gap
    run_boundary = np.zeros(len(sleep_win), dtype=bool)
    run_boundary[0] = True
    run_boundary[1:] = (sleep_win[1:] != sleep_win[:-1]) | gap_break[1:]
    run_id = np.cumsum(run_boundary)

    # Extract start/end times of True runs
    sleep_idx = np.where(sleep_win)[0]
    if len(sleep_idx) == 0:
        return pd.DataFrame(columns=["start", "end", "duration"])

    sleep_runs = run_id[sleep_idx]
    unique_runs, first_pos = np.unique(sleep_runs, return_index=True)
    last_pos = np.concatenate([first_pos[1:] - 1, [len(sleep_idx) - 1]])

    start_times = dt[sleep_idx[first_pos]]
    end_times = dt[sleep_idx[last_pos]]

    # --- 5. merge gaps < max_gap_minutes (single pass) ---
    gap = np.timedelta64(max_gap_minutes, "m")
    ms, me = [start_times[0]], [end_times[0]]
    for i in range(1, len(start_times)):
        if start_times[i] - me[-1] <= gap:
            me[-1] = end_times[i]
        else:
            ms.append(start_times[i])
            me.append(end_times[i])
    ms, me = np.array(ms), np.array(me)

    # --- 6. filter minimum duration ---
    dur_s = (me - ms) / np.timedelta64(1, "s")
    keep = dur_s >= min_sleep_minutes * 60

    periods = pd.DataFrame({
        "start": pd.to_datetime(ms[keep]),
        "end": pd.to_datetime(me[keep]),
    })
    if periods.empty:
        return periods

    # --- 7. annotate ---
    periods["duration"] = (
        (periods["end"] - periods["start"]).dt.total_seconds() / 3600
    ).round(2)
    return annotate_periods(periods, df, _hrv_dts, _hrv_vals)


gravity_sleep = detect_sleep_from_gravity(df)
gravity_sleep

In [ ]:
# Compare gravity-detected sleep vs ppg_green ground truth
print(f"ppg_green (ground truth): {len(sleep_df)} periods")
display(sleep_df[["start", "end", "duration"]].tail(20))
print(f"\ngravity detected: {len(gravity_sleep)} periods")
display(gravity_sleep[["start", "end", "duration"]].tail(20))

# Overlap check (last 30 gravity periods to avoid excessive output)
print("\nOverlap (last 30 gravity periods):")
recent_gravity = gravity_sleep.tail(30)
# Only check ppg periods that could overlap with the recent gravity window
if not recent_gravity.empty:
    window_start = recent_gravity["start"].iloc[0]
    relevant_ppg = sleep_df[sleep_df["end"] >= window_start]
    for _, grow in recent_gravity.iterrows():
        for _, prow in relevant_ppg.iterrows():
            overlap_start = max(grow["start"], prow["start"])
            overlap_end   = min(grow["end"],   prow["end"])
            overlap_h = max(pd.Timedelta(0), overlap_end - overlap_start).total_seconds() / 3600
            if overlap_h > 0:
                print(f"  gravity {grow['start'].strftime('%m-%d %H:%M')}\u2013{grow['end'].strftime('%H:%M')} "
                      f"\u2194 ppg {prow['start'].strftime('%m-%d %H:%M')}\u2013{prow['end'].strftime('%H:%M')}  "
                      f"overlap={overlap_h:.2f}h")

In [ ]:

def plot_with_sleep_annotations(days, df_signal, gravity_periods, ppg_periods, plot_count=2):
    """
    Plot heart rate + gravity_delta_smooth per day with shaded sleep regions:
      - Blue fill  : gravity-detected sleep
      - Red dashed : ppg_green ground-truth sleep
    """
    for index, day_data in enumerate(days):
        day_start = day_data["datetime"].iloc[0]
        day_end   = day_data["datetime"].iloc[-1]

        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(x=day_data["datetime"], y=day_data["bpm"],
                       mode="lines", name="Heart Rate (BPM)", line=dict(color="blue")),
            secondary_y=False,
        )
        fig.add_trace(
            go.Scatter(x=day_data["datetime"], y=day_data["gravity_delta_smooth"],
                       mode="lines", name="gravity_delta_smooth", line=dict(color="orange")),
            secondary_y=True,
        )

        # gravity-detected sleep — solid blue fill
        for _, row in gravity_periods.iterrows():
            s, e = max(row["start"], day_start), min(row["end"], day_end)
            if s < e:
                fig.add_vrect(x0=s, x1=e,
                              fillcolor="blue", opacity=0.12, line_width=0,
                              annotation_text="gravity sleep", annotation_position="top left")

        # ppg_green ground-truth — red dashed border, no fill
        for _, row in ppg_periods.iterrows():
            s, e = max(row["start"], day_start), min(row["end"], day_end)
            if s < e:
                fig.add_vrect(x0=s, x1=e,
                              fillcolor="rgba(0,0,0,0)",
                              line=dict(color="red", width=2, dash="dash"),
                              annotation_text="ppg sleep", annotation_position="top right")

        fig.update_layout(
            title=f"Sleep detection — {day_start.date()} noon to next noon",
            xaxis_title="Time",
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        )
        fig.update_xaxes(tickformat="%H:%M")
        fig.update_yaxes(title_text="Heart Rate (BPM)", secondary_y=False)
        fig.update_yaxes(title_text="gravity_delta_smooth (g)", secondary_y=True)
        fig.show()

        if index == plot_count:
            break


plot_with_sleep_annotations(
    filter_noon_to_noon(df), df,
    gravity_periods=gravity_sleep,
    ppg_periods=sleep_df,
)
